# Solar filament segmentation: full inference pipeline

This notebook reproduces our final submission (public leaderboard 0.40) from the committed weights, with no training and no downloads:

| Stage | What | Weights |
|---|---|---|
| 1 | YOLO11m-seg at 1536 px, three passes (identity, horizontal flip, vertical flip), soft masks | `weights/yolo11m_seg_1536.pt` |
| 2 | Instance fusion over the three passes (IoU >= 0.5, mean score) | - |
| 3a | U-Net (ResNet34) full-resolution crop refiner | `weights/unet_refiner.pt` |
| 3b | SAM ViT-B + LoRA crop refiner | `weights/sam_vit_base/` (base, Apache-2.0) + `weights/sam_lora_refiner.pt` |
| 4 | 50/50 soft-mask blend of the two refiners | - |
| 5 | Calibrated decoding: conf 0.25, mask threshold 0.35, min area 100, NMS 0.3, max 20 per image, pixel-wise overlap resolution, RLE | `configs/calib.json` |

Requirements: `pip install -r requirements.txt` (Python 3.12) and a CUDA GPU (we used a 16 GB RTX 4080 SUPER). The competition test images go in `data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images` (or set `IMAGES` below).

## 0. Setup

In [ ]:
import os, sys, json, time
from pathlib import Path
os.environ.setdefault('HF_HUB_OFFLINE', '1')  # everything needed is in the repo

REPO = Path('..').resolve() if Path('../scripts/run_pipeline.py').exists() else Path('.').resolve()
sys.path[:0] = [str(REPO / 'src'), str(REPO / 'scripts')]
os.chdir(REPO)

IMAGES = Path(os.environ.get('FILSEG_IMAGES', 'data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images'))
WORKDIR = Path('work'); WORKDIR.mkdir(exist_ok=True)
OUT = Path('submission.csv')
DEVICE = 'cuda'

import cv2, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from run_pipeline import DEFAULTS, refine_cache, blend
from filseg.infer import predict_raw, save_cache, load_cache
from filseg.ensemble import ensemble_caches
from filseg.postprocess import PPParams, group_by_stem, group_soft_by_stem, postprocess_to_rows
from filseg.submission import write_submission, validate_submission
from filseg.rle import decode_rle

paths = sorted(str(p) for p in IMAGES.glob('*.jpeg'))
images = {Path(p).stem: p for p in paths}
print(len(paths), 'test images;', 'torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print(DEFAULTS)

## 1. Detector: three YOLO11m-seg passes with soft masks

In [ ]:
t0 = time.time()
passes = {'': 'raw.pkl', 'h': 'raw_h.pkl', 'v': 'raw_v.pkl'}
for flip, name in passes.items():
    df = predict_raw(DEFAULTS['yolo'], paths, 1536, 0.01, 0.7, 100, 2, soft=True, flip=flip)
    save_cache(df, str(WORKDIR / name))
    print(f"pass flip={flip or 'none':4s}: {len(df)} candidate instances")
print(f'{time.time() - t0:.0f}s')

## 2. Instance fusion over the three passes

In [ ]:
fused = ensemble_caches([load_cache(str(WORKDIR / n)) for n in passes.values()], 0.5, 1, 8)
save_cache(fused, str(WORKDIR / 'flip3.pkl'))
print(len(fused), 'fused instances')

## 3. Crop refiners (U-Net and SAM ViT-B + LoRA)
Each fused instance is cropped at full resolution around its soft mask; the refiner sees the image crop and the prior and predicts a new mask.

In [ ]:
for key, batch in (('unet', 64), ('sam', 2)):
    t0 = time.time()
    df = refine_cache(fused, DEFAULTS[key], images, batch, DEVICE)
    save_cache(df, str(WORKDIR / f'{key}.pkl'))
    print(f'{key}: {len(df)} refined instances in {time.time() - t0:.0f}s')

## 4. 50/50 blend of the two refiners

In [ ]:
blend(WORKDIR / 'sam.pkl', WORKDIR / 'unet.pkl', WORKDIR / 'blend.pkl')
raw = load_cache(str(WORKDIR / 'blend.pkl'))
print(len(raw), 'blended instances')

## 5. Decoding and submission

In [ ]:
bp = json.load(open(DEFAULTS['calib']))['best_params']
params = PPParams(bp['conf'], bp['min_area'], bp['max_det'], bp['nms'], bp.get('erode', 0),
                  bp.get('erode_shape', 'square'), bp.get('mask_thr'))
print(params)
stems = sorted(images)
sub = write_submission(postprocess_to_rows(group_by_stem(raw), params, stems, group_soft_by_stem(raw)), str(OUT))
rep = validate_submission(str(OUT), set(stems))
print('valid:', rep.ok, '| rows', rep.n_rows, '| images', rep.n_stems, '/', len(stems), '| overlapping pairs', rep.n_overlap_pairs)
sub.head()

## 6. Predictions on a few test images

In [ ]:
sub['stem'] = sub['filament_id'].str.rsplit('_', n=1).str[0]
show = [s for s in stems if s in set(sub['stem'])][::45][:4]
fig, axs = plt.subplots(1, len(show), figsize=(5 * len(show), 5))
for ax, st in zip(np.atleast_1d(axs), show):
    img = cv2.cvtColor(cv2.imread(images[st], cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2RGB)
    for r in sub.loc[sub['stem'] == st, 'segmentation_rle']:
        m = decode_rle(r).astype(np.uint8)
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(img, cs, -1, (255, 0, 255), 6)
    ax.imshow(img); ax.set_title(f'{st}: {int((sub.stem == st).sum())} filaments'); ax.axis('off')
plt.tight_layout(); plt.show()